# 예제 03. 순전파와 역전파
빅데이터프로그래밍 · 6주차

## 목표
- 순전파에서 값이 흐르는 것을 확인한다
- 역전파로 각 층의 기울기가 채워지는 것을 확인한다
- 4주차 자동미분과 5주차 학습 루프가 여기서 만난다


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
)
x = torch.randn(6, 3)
y = torch.randn(6, 1)


## 1. 순전파 — 입력에서 출력으로


In [ ]:
pred = model(x)
print("예측:", pred.shape)
print(pred.data.round(decimals=3).flatten())


## 2. 손실 계산 — 얼마나 틀렸는가


In [ ]:
loss_fn = nn.MSELoss()
loss = loss_fn(pred, y)
print("손실:", loss.item())


## 3. 역전파 전 — 기울기는 비어 있습니다


In [ ]:
for name, p in model.named_parameters():
    print(f"{name:12s} grad = {p.grad}")


## 4. 역전파 후 — 모든 층에 기울기가 채워집니다


In [ ]:
loss.backward()

for name, p in model.named_parameters():
    print(f"{name:12s} grad shape {tuple(p.grad.shape)}  "
          f"평균 {p.grad.mean().item():+.5f}")


한 번의 `backward()` 로 **모든 층의 기울기가 한꺼번에** 계산됩니다.
출력층에서 입력층 방향으로 거꾸로 전달되기 때문에 역전파라고 부릅니다.


## 5. 기울기의 방향이 뜻하는 것
기울기가 양수면 그 값을 줄여야 손실이 줄어듭니다. 반대 방향으로 움직이는 것이 경사하강법입니다.


In [ ]:
w = model[0].weight
print("현재 값 (일부):", w.data[0].round(decimals=4))
print("기울기 (일부):", w.grad[0].round(decimals=4))

lr = 0.1
with torch.no_grad():
    w -= lr * w.grad                       # 기울기 반대 방향으로
print("갱신 후 (일부):", w.data[0].round(decimals=4))


## 6. 학습 루프에 넣으면 — 5주차 다섯 단계


In [ ]:
model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 1))
opt = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(50):
    pred = model(x)                        # 1. 순전파
    loss = loss_fn(pred, y)                # 2. 손실

    opt.zero_grad()                        # 3. 초기화
    loss.backward()                        # 4. 역전파
    opt.step()                             # 5. 갱신

    if epoch % 10 == 0 or epoch == 49:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")


## 7. 순전파만 하고 싶을 때
예측만 할 때는 기울기를 만들지 않습니다.


In [ ]:
model.eval()
with torch.no_grad():
    out = model(x)
print("requires_grad:", out.requires_grad)


## 직접 해보기
1. 은닉층을 두 개로 늘린 뒤 `backward()` 후 기울기가 몇 개 층에 생기는지 확인하세요.
2. `opt.zero_grad()` 를 빼면 손실이 어떻게 달라지나요?


In [ ]:
# 여기에 작성하세요
